# 08 - Segmentation Model v2 (UNet++ + Albumentations + FocalDice)

Cải tiến so với notebook 05:
- **Dataset**: pseudo labels v2 từ notebook 07 (~3000 samples thay vì 500)
- **Model**: `UNet++` (segmentation-models-pytorch) thay EfficientNetUNet tự build
- **Augmentation**: Albumentations mạnh (ElasticTransform, GridDistortion, ColorJitter...)
- **Loss**: `FocalDice` (Focal + Dice) thay BCE + Dice — tốt hơn cho class imbalance
- **Optimizer**: `AdamW` + `CosineAnnealingLR` (50 epochs)
- **Benchmark**: So sánh v1 vs v2 tại cuối notebook

In [1]:
# Cài đặt dependencies nếu chưa có
import subprocess, sys

def ensure_installed(pip_name, import_name=None):
    import_name = import_name or pip_name.replace('-', '_')
    try:
        __import__(import_name)
        print(f'[OK] {pip_name}')
    except ImportError:
        print(f'Installing {pip_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name, '-q'])
        print(f'[Installed] {pip_name}')

ensure_installed('segmentation-models-pytorch', 'segmentation_models_pytorch')
ensure_installed('albumentations')

[OK] segmentation-models-pytorch
[OK] albumentations


In [2]:
import os
import sys
import json
import random

notebook_dir = os.path.abspath('')
proj_root = os.path.dirname(notebook_dir) if os.path.basename(notebook_dir) == 'notebooks' else notebook_dir
sys.path.insert(0, proj_root)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('proj_root:', proj_root)

device: cuda
proj_root: e:\Master\thesis-clean


## 1) Dataset với Albumentations augmentation

In [3]:
IMAGE_DIR = os.path.join(proj_root, 'notebooks', 'data', 'processed_v2_train')
MASK_DIR  = os.path.join(proj_root, 'notebooks', 'data', 'pseudo_labels_v2_train')

if not os.path.exists(IMAGE_DIR):
    raise FileNotFoundError(
        f'Không tìm thấy {IMAGE_DIR}.\nChạy notebook 07-pseudo-labeling-v2 trước.'
    )

print('IMAGE_DIR:', IMAGE_DIR, '| exists:', os.path.exists(IMAGE_DIR))
print('MASK_DIR :', MASK_DIR,  '| exists:', os.path.exists(MASK_DIR))

sample_masks = [f for f in os.listdir(MASK_DIR) if f.endswith('_pseudo.png')][:3]
print('Sample masks:', sample_masks)

IMAGE_DIR: e:\Master\thesis-clean\notebooks\data\processed_v2_train | exists: True
MASK_DIR : e:\Master\thesis-clean\notebooks\data\pseudo_labels_v2_train | exists: True
Sample masks: ['ALGAL_LEAF_SPOT_to_label_1.jpg_pseudo.png', 'ALGAL_LEAF_SPOT_to_label_10.jpg_pseudo.png', 'ALGAL_LEAF_SPOT_to_label_101.jpg_pseudo.png']


In [4]:
# Albumentations transforms — synchronized image + mask

TRAIN_TRANSFORM = A.Compose([
    A.RandomResizedCrop(size=(224, 224), scale=(0.7, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=30, p=0.5),
    A.ElasticTransform(p=0.3),
    A.GridDistortion(p=0.2),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.5),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
], additional_targets={'mask': 'mask'})

VAL_TRANSFORM = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
], additional_targets={'mask': 'mask'})

print('Transforms defined')

Transforms defined


e:\Master\thesis-clean\venv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
class DurianSegDatasetV2(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.transform = transform
        self.samples   = []

        for fname in sorted(os.listdir(mask_dir)):
            if not fname.endswith('_pseudo.png'):
                continue
            img_name  = fname.replace('_pseudo.png', '')
            img_path  = os.path.join(image_dir, img_name)
            mask_path = os.path.join(mask_dir, fname)
            if os.path.exists(img_path):
                self.samples.append((img_path, mask_path))

        print(f'[Dataset] {len(self.samples)} image-mask pairs')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        img  = np.array(Image.open(img_path).convert('RGB'))
        mask = np.array(Image.open(mask_path).convert('L'))
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug['image']                     # [3, H, W] tensor
            mask = aug['mask'].unsqueeze(0).float() # [1, H, W] tensor
        else:
            img  = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
            mask = torch.from_numpy(mask).unsqueeze(0).float()

        return {'image': img, 'label': mask, 'img_path': img_path}

In [6]:
# Tạo 3 dataset instances với transform khác nhau (cùng samples)
train_dataset    = DurianSegDatasetV2(IMAGE_DIR, MASK_DIR, transform=TRAIN_TRANSFORM)
val_test_dataset = DurianSegDatasetV2(IMAGE_DIR, MASK_DIR, transform=VAL_TRANSFORM)

n       = len(train_dataset)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val
print(f'Total: {n} | train={n_train} val={n_val} test={n_test}')

# Deterministic shuffle — cùng seed với nb05 để split có thể so sánh
rng     = np.random.default_rng(42)
shuffled = rng.permutation(n).tolist()
train_idx = shuffled[:n_train]
val_idx   = shuffled[n_train:n_train + n_val]
test_idx  = shuffled[n_train + n_val:]

train_set = Subset(train_dataset,    train_idx)
val_set   = Subset(val_test_dataset, val_idx)
test_set  = Subset(val_test_dataset, test_idx)

train_loader = DataLoader(train_set, batch_size=8, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=8, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=8, shuffle=False, num_workers=0)

batch = next(iter(train_loader))
print('image shape:', batch['image'].shape)
print('label shape:', batch['label'].shape)

[Dataset] 3104 image-mask pairs
[Dataset] 3104 image-mask pairs
Total: 3104 | train=2172 val=465 test=467
image shape: torch.Size([8, 3, 224, 224])
label shape: torch.Size([8, 1, 224, 224])


## 2) Model — UNet++ với EfficientNet-B0 encoder

UNet++ tốt hơn UNet chuẩn cho spot disease vì:
- Dense skip connections (nested architecture)
- Re-designed skip pathways giúp giảm semantic gap giữa encoder/decoder
- Tốt hơn cho multiple small instances (nhiều đốm nhỏ)

In [7]:
model = smp.UnetPlusPlus(
    encoder_name='efficientnet-b0',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None,  # logits output, sigmoid trong loss
)
model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'UNet++ EfficientNet-B0')
print(f'  Total params    : {total_params:,}')
print(f'  Trainable params: {trainable_params:,}')

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

e:\Master\thesis-clean\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pc\.cache\huggingface\hub\models--smp-hub--efficientnet-b0.imagenet. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

UNet++ EfficientNet-B0
  Total params    : 6,569,581
  Trainable params: 6,569,581


## 3) Loss + Optimizer + Scheduler

- **FocalDice**: Focal loss giải quyết class imbalance (background >> foreground trong ảnh đốm)
- **AdamW**: Adam + weight decay (tốt hơn Adam cho regularization)
- **CosineAnnealing**: smooth LR decay, không bị stuck như ReduceLROnPlateau

In [8]:
class FocalDiceLoss(nn.Module):
    """Focal Loss + Dice Loss cho binary segmentation với class imbalance."""

    def __init__(self, alpha=0.25, gamma=2.0, smooth=1.0):
        super().__init__()
        self.alpha  = alpha
        self.gamma  = gamma
        self.smooth = smooth

    def focal_loss(self, pred, target):
        bce   = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        p_t   = torch.exp(-bce)
        focal = self.alpha * (1.0 - p_t) ** self.gamma * bce
        return focal.mean()

    def dice_loss(self, pred, target):
        pred  = torch.sigmoid(pred)
        inter = (pred * target).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice  = (2.0 * inter + self.smooth) / (union + self.smooth)
        return (1.0 - dice).mean()

    def forward(self, pred, target):
        return self.focal_loss(pred, target) + self.dice_loss(pred, target)


EPOCHS    = 50
criterion = FocalDiceLoss(alpha=0.25, gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f'Loss     : FocalDice (alpha=0.25, gamma=2.0)')
print(f'Optimizer: AdamW (lr=1e-4, wd=1e-4)')
print(f'Scheduler: CosineAnnealing (T_max={EPOCHS})')
print(f'Epochs   : {EPOCHS}')

Loss     : FocalDice (alpha=0.25, gamma=2.0)
Optimizer: AdamW (lr=1e-4, wd=1e-4)
Scheduler: CosineAnnealing (T_max=50)
Epochs   : 50


## 4) Metrics

In [9]:
def calculate_metrics(pred_np, gt_np, smooth=1e-6):
    pred = pred_np.flatten().astype(bool)
    gt   = gt_np.flatten().astype(bool)
    tp   = (pred & gt).sum()
    fp   = (pred & ~gt).sum()
    fn   = (~pred & gt).sum()
    precision = (tp + smooth) / (tp + fp + smooth)
    recall    = (tp + smooth) / (tp + fn + smooth)
    f1        = 2 * precision * recall / (precision + recall + smooth)
    dice      = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou       = (tp + smooth) / (tp + fp + fn + smooth)
    return {'iou': float(iou), 'dice': float(dice),
            'precision': float(precision), 'recall': float(recall), 'f1': float(f1)}


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_m = {'iou': [], 'dice': [], 'precision': [], 'recall': [], 'f1': []}

    with torch.no_grad():
        for batch in loader:
            imgs  = batch['image'].to(device)
            masks = batch['label'].to(device)
            out   = model(imgs)
            loss  = criterion(out, masks)
            total_loss += loss.item() * imgs.size(0)

            preds = (torch.sigmoid(out) > 0.5).cpu().numpy().astype(np.uint8)
            gts   = masks.cpu().numpy().astype(np.uint8)
            for i in range(len(preds)):
                m = calculate_metrics(preds[i, 0], gts[i, 0])
                for k in all_m:
                    all_m[k].append(m[k])

    avg_loss = total_loss / len(loader.dataset)
    avg_m    = {k: float(np.mean(v)) for k, v in all_m.items()}
    return avg_loss, avg_m


print('Metrics functions ready')

Metrics functions ready


## 5) Training loop — 50 epochs

In [ ]:
ckpt_dir_v2    = os.path.join(proj_root, 'notebooks', 'models', 'segmentation_v2', 'checkpoints')
metrics_dir_v2 = os.path.join(proj_root, 'notebooks', 'models', 'segmentation_v2', 'metrics')
viz_dir_v2     = os.path.join(proj_root, 'notebooks', 'models', 'segmentation_v2', 'visualizations')
for d in [ckpt_dir_v2, metrics_dir_v2, viz_dir_v2]:
    os.makedirs(d, exist_ok=True)

PATIENCE      = 10
best_val_loss = float('inf')
patience_cnt  = 0
history = {
    'train_loss': [], 'train_iou': [], 'train_dice': [],
    'val_loss':   [], 'val_iou':   [], 'val_dice':   [],
    'val_precision': [], 'val_recall': [], 'val_f1': [],
    'lr': [],
}

for epoch in range(1, EPOCHS + 1):
    # ---- TRAIN ----
    model.train()
    train_loss = 0.0
    tp = fp = fn = 0

    for batch in tqdm(train_loader, desc=f'E{epoch:02d}/{EPOCHS} [Train]', leave=False):
        imgs  = batch['image'].to(device)
        masks = batch['label'].to(device)

        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        preds = torch.sigmoid(out) > 0.5
        tp   += ((preds == 1) & (masks == 1)).sum().item()
        fp   += ((preds == 1) & (masks == 0)).sum().item()
        fn   += ((preds == 0) & (masks == 1)).sum().item()

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    train_loss /= len(train_loader.dataset)
    train_iou   = tp / (tp + fp + fn + 1e-7)
    train_dice  = 2 * tp / (2 * tp + fp + fn + 1e-7)

    # ---- VALIDATION ----
    val_loss, val_m = evaluate(model, val_loader, criterion, device)

    # ---- HISTORY ----
    history['train_loss'].append(train_loss)
    history['train_iou'].append(train_iou)
    history['train_dice'].append(train_dice)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_m['iou'])
    history['val_dice'].append(val_m['dice'])
    history['val_precision'].append(val_m['precision'])
    history['val_recall'].append(val_m['recall'])
    history['val_f1'].append(val_m['f1'])
    history['lr'].append(current_lr)

    if epoch % 5 == 0 or epoch <= 3:
        print(
            f'E{epoch:02d} | '
            f'train_loss={train_loss:.4f} val_loss={val_loss:.4f} | '
            f'train_iou={train_iou:.4f} val_iou={val_m["iou"]:.4f} | '
            f'val_dice={val_m["dice"]:.4f} | lr={current_lr:.2e}'
        )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_cnt  = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_loss': val_loss,
            'val_metrics': val_m,
        }, os.path.join(ckpt_dir_v2, 'unetpp_best.pth'))
        if epoch % 5 == 0 or epoch <= 3:
            print(f'  Best model saved (val_loss={best_val_loss:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'\nEarly stop at epoch {epoch} (no val_loss improvement for {PATIENCE} epochs).')
            break

with open(os.path.join(metrics_dir_v2, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)
print('\nTraining complete. History saved.')

## 6) Đánh giá trên tập test

In [13]:
best_ckpt = os.path.join(ckpt_dir_v2, 'unetpp_best.pth')
ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded best model from epoch {ckpt["epoch"]} (val_loss={ckpt["val_loss"]:.4f})')

test_loss, test_m = evaluate(model, test_loader, criterion, device)

print('\n=== Test Set Metrics (V2) ===')
print(f'  Loss     : {test_loss:.4f}')
for k, v in test_m.items():
    print(f'  {k:<10}: {v:.4f}')

test_results_v2 = {'test_loss': test_loss, **test_m}
with open(os.path.join(metrics_dir_v2, 'test_metrics.json'), 'w') as f:
    json.dump(test_results_v2, f, indent=2)
print('\nTest metrics saved.')

RuntimeError: CUDA error: the launch timed out and was terminated
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 7) Benchmark — V1 (nb05) vs V2 (nb08)

In [ ]:
v1_metrics_path = os.path.join(proj_root, 'notebooks', 'models', 'segmentation', 'metrics', 'test_metrics.json')

print('=' * 60)
print('BENCHMARK: V1 (EfficientNet+UNet, nb05) vs V2 (UNet++, nb08)')
print('=' * 60)

if os.path.exists(v1_metrics_path):
    with open(v1_metrics_path) as f:
        v1_m = json.load(f)

    metric_keys = ['iou', 'dice', 'precision', 'recall', 'f1']
    print(f'\n{"Metric":<12} {"V1 Baseline":>14} {"V2 Improved":>14} {"Delta":>10} {"Δ%":>8}')
    print('-' * 62)

    for k in metric_keys:
        v1_val = v1_m.get(k, 0.0)
        v2_val = test_m.get(k, 0.0)
        delta  = v2_val - v1_val
        pct    = (delta / (v1_val + 1e-9)) * 100
        sign   = '+' if delta >= 0 else ''
        print(f'{k:<12} {v1_val:>14.4f} {v2_val:>14.4f} {sign+f"{delta:.4f}":>10} {sign+f"{pct:.1f}%":>8}')

    print('\n--- Training Data ---')
    v1_hist_path = os.path.join(proj_root, 'notebooks', 'models', 'segmentation', 'metrics', 'training_history.json')
    if os.path.exists(v1_hist_path):
        with open(v1_hist_path) as f:
            v1_hist = json.load(f)
        print(f'  V1 best val IoU : {max(v1_hist["val_iou"]):.4f}  (epochs={len(v1_hist["val_iou"])})')
    print(f'  V2 best val IoU : {max(history["val_iou"]):.4f}  (epochs={len(history["val_iou"])})')

else:
    print('V1 metrics not found. Chạy notebook 05 trước để benchmark.')
    print('\nV2 Test Results:')
    for k, v in test_results_v2.items():
        print(f'  {k:<12}: {v:.4f}')

print('\n--- V2 Improvements Summary ---')
print(f'  Pseudo labels    : ~{n} samples (v1: 500)')
print(f'  CAM method       : GradCAM++ multi-scale (v1: GradCAM single-layer)')
print(f'  Threshold        : Otsu adaptive (v1: fixed 0.5)')
print(f'  Model            : UNet++ EfficientNet-B0 (v1: EfficientNet+UNet custom)')
print(f'  Augmentation     : Albumentations strong (v1: flip only)')
print(f'  Loss             : FocalDice (v1: BCE+Dice)')
print(f'  Optimizer        : AdamW + CosineAnnealing (v1: Adam + ReduceLROnPlateau)')
print(f'  Early stopping   : patience={PATIENCE} epochs')
print(f'  Epochs trained   : {len(history["val_loss"])} / {EPOCHS}')

## 8) Training curves

In [ ]:
eps = range(1, EPOCHS + 1)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].plot(eps, history['train_loss'], label='Train', color='royalblue')
axes[0, 0].plot(eps, history['val_loss'],   label='Val',   color='tomato')
axes[0, 0].set_title('Loss (FocalDice)')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(eps, history['train_iou'], label='Train', color='royalblue')
axes[0, 1].plot(eps, history['val_iou'],   label='Val',   color='tomato')
axes[0, 1].set_title('IoU')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(eps, history['train_dice'], label='Train', color='royalblue')
axes[0, 2].plot(eps, history['val_dice'],   label='Val',   color='tomato')
axes[0, 2].set_title('Dice')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

axes[1, 0].plot(eps, history['val_precision'], label='Precision', color='green')
axes[1, 0].plot(eps, history['val_recall'],    label='Recall',    color='orange')
axes[1, 0].plot(eps, history['val_f1'],        label='F1',        color='purple')
axes[1, 0].set_title('Val Precision / Recall / F1')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(eps, history['lr'], color='gray')
axes[1, 1].set_title('Learning Rate (CosineAnnealing)')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

# V1 vs V2 bar chart (if v1 available)
if os.path.exists(v1_metrics_path):
    metric_keys = ['iou', 'dice', 'f1']
    x = np.arange(len(metric_keys))
    w = 0.35
    v1_vals = [v1_m.get(k, 0) for k in metric_keys]
    v2_vals = [test_m.get(k, 0) for k in metric_keys]
    axes[1, 2].bar(x - w/2, v1_vals, w, label='V1 Baseline', color='lightcoral')
    axes[1, 2].bar(x + w/2, v2_vals, w, label='V2 Improved', color='steelblue')
    axes[1, 2].set_xticks(x)
    axes[1, 2].set_xticklabels([k.upper() for k in metric_keys])
    axes[1, 2].set_title('Test Metrics: V1 vs V2')
    axes[1, 2].set_ylim(0, 1)
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3, axis='y')
else:
    axes[1, 2].axis('off')

plt.suptitle('Segmentation V2 Training (UNet++ + FocalDice + Albumentations)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(viz_dir_v2, 'training_curves_v2.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Training curves saved.')

## 9) Sample predictions trên test set

In [ ]:
model.eval()
sample_batch = next(iter(test_loader))
imgs     = sample_batch['image'].to(device)
masks_gt = sample_batch['label']

with torch.no_grad():
    outputs = model(imgs)
    probs   = torch.sigmoid(outputs)
    preds   = (probs > 0.5).float()

n_show = min(4, len(imgs))
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
if n_show == 1:
    axes = axes.reshape(1, -1)

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

for i in range(n_show):
    img_denorm  = (imgs[i].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    gt_mask     = masks_gt[i, 0].numpy()
    pred_mask   = preds[i, 0].cpu().numpy()
    prob_map    = probs[i, 0].cpu().numpy()

    m = calculate_metrics(pred_mask, gt_mask)

    axes[i, 0].imshow(img_denorm)
    axes[i, 0].set_title('Input Image', fontsize=9)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(gt_mask, cmap='gray')
    axes[i, 1].set_title('Pseudo Label (v2)', fontsize=9)
    axes[i, 1].axis('off')

    axes[i, 2].imshow(prob_map, cmap='hot', vmin=0, vmax=1)
    axes[i, 2].set_title('Probability Map', fontsize=9)
    axes[i, 2].axis('off')

    axes[i, 3].imshow(pred_mask, cmap='gray')
    axes[i, 3].set_title(f'Prediction\nIoU={m["iou"]:.3f} Dice={m["dice"]:.3f}', fontsize=9)
    axes[i, 3].axis('off')

plt.suptitle('UNet++ V2 Test Predictions', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(viz_dir_v2, 'test_predictions_v2.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Sample predictions saved.')